# Phase 2: Reranker Fine-Tuning with QLoRA

This notebook fine-tunes the project's exact base reranker (**`BAAI/bge-reranker-base`**) using **4-bit QLoRA (PEFT + BitsAndBytes)** on the enterprise RAG dataset.

### Pipeline Context:
$$\text{Query} \rightarrow \text{Baseline Embeddings} \rightarrow \text{Hybrid Retrieval} \rightarrow \mathbf{\text{Fine-Tuned Reranker}} \rightarrow \text{Baseline LLM} \rightarrow \text{Answer}$$

---

## 1. Environment Check

In [ ]:
import os
import sys
import json
import random
import shutil
import time
import math
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional, Any

import torch
import numpy as np

print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)
print(f"Python Version:       {sys.version.split()[0]}")
print(f"PyTorch Version:      {torch.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:          {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Total VRAM:           {props.total_memory / (1024**3):.2f} GB")
else:
    print("[WARNING] CUDA is not available. Ensure GPU accelerator is enabled in Kaggle Settings.")
print("=" * 70)

## 2. Install Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers>=4.44.0 peft>=0.12.0 bitsandbytes>=0.43.0 accelerate>=0.33.0 datasets>=2.20.0 rank-bm25>=0.2.2 sentence-transformers>=3.0.0 safetensors

import transformers
import peft
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)
from rank_bm25 import BM25Okapi

print(f"Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__}")
print(f"BitsAndBytes: {bnb.__version__}")

## 3. Hyperparameters & Configuration

In [ ]:
@dataclass
class HyperParameters:
    base_model_name: str = "BAAI/bge-reranker-base"
    max_length: int = 512
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: Tuple[str, ...] = ("query", "key", "value", "dense")
    
    questions_file: str = "questions_visible.jsonl"
    chunks_file: str = "chunks.jsonl"
    
    num_train_epochs: int = 3
    learning_rate: float = 2e-4
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size: int = 16
    gradient_accumulation_steps: int = 2
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    seed: int = 42
    num_negatives_per_positive: int = 4
    
    train_ratio: float = 0.80
    val_ratio: float = 0.10
    test_ratio: float = 0.10
    
    output_dir: str = "/kaggle/working/reranker_phase2_output"
    package_dir: str = "/kaggle/working/reranker_phase2"

CONFIG = HyperParameters()

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG.seed)
print("Configuration:", json.dumps(CONFIG.__dict__, indent=2))

## 4. Load Current Dataset & Sanity Checks

In [ ]:
def locate_file(filename: str) -> str:
    search_paths = [
        filename,
        os.path.join(".", filename),
        os.path.join("..", filename),
        os.path.join("/kaggle/input", filename),
        os.path.join("/kaggle/input/enterprise-rag-dataset", filename),
        os.path.join("/kaggle/working", filename),
    ]
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            if filename in files:
                return os.path.join(root, filename)
    for p in search_paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Could not locate '{filename}'. Upload as Kaggle dataset or put in working directory.")

q_path = locate_file(CONFIG.questions_file)
c_path = locate_file(CONFIG.chunks_file)

questions_data = []
with open(q_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            questions_data.append(json.loads(line))

chunks_dict = {}
with open(c_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rec = json.loads(line)
            chunks_dict[rec["chunk_id"]] = rec

print(f"Total Questions: {len(questions_data)}")
print(f"Total Chunks:    {len(chunks_dict)}")

## 5. Convert Dataset for Reranker Training & Mine Hard Negatives

In [ ]:
chunk_id_list = list(chunks_dict.keys())
corpus_texts = [chunks_dict[cid]["text"] for cid in chunk_id_list]
tokenized_corpus = [text.lower().split() for text in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

processed_qa_samples = []
for item in questions_data:
    qid = item["id"]
    query = item["question"]
    gold_cids = set(item.get("relevant_chunk_ids", []))
    
    pos_texts = [chunks_dict[cid]["text"] for cid in gold_cids if cid in chunks_dict]
    if not pos_texts:
        continue
        
    query_tokens = query.lower().split()
    top_indices = bm25.get_top_n(query_tokens, list(range(len(chunk_id_list))), n=35)
    
    neg_texts = []
    for idx in top_indices:
        candidate_cid = chunk_id_list[idx]
        if candidate_cid not in gold_cids:
            neg_texts.append(chunks_dict[candidate_cid]["text"])
        if len(neg_texts) >= CONFIG.num_negatives_per_positive * len(pos_texts):
            break
            
    processed_qa_samples.append({
        "id": qid,
        "query": query,
        "pos_texts": pos_texts,
        "neg_texts": neg_texts,
    })

print(f"Created {len(processed_qa_samples)} query samples with hard negatives.")

## 6. Query-Level Train / Validation / Test Split

In [ ]:
random.seed(CONFIG.seed)
random.shuffle(processed_qa_samples)

n_total = len(processed_qa_samples)
n_train = int(n_total * CONFIG.train_ratio)
n_val = int(n_total * CONFIG.val_ratio)

train_queries = processed_qa_samples[:n_train]
val_queries   = processed_qa_samples[n_train:n_train + n_val]
test_queries  = processed_qa_samples[n_train + n_val:]

def flatten_to_pairs(query_records):
    pairs = []
    for rec in query_records:
        for pos in rec["pos_texts"]:
            pairs.append({"query": rec["query"], "passage": pos, "label": 1.0})
        for neg in rec["neg_texts"]:
            pairs.append({"query": rec["query"], "passage": neg, "label": 0.0})
    return pairs

train_pairs = flatten_to_pairs(train_queries)
val_pairs   = flatten_to_pairs(val_queries)
test_pairs  = flatten_to_pairs(test_queries)

print(f"Train Pairs: {len(train_pairs)} | Val Pairs: {len(val_pairs)} | Test Pairs: {len(test_pairs)}")

## 7. Load Base Reranker & 8. Apply QLoRA

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG.base_model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG.base_model_name,
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
)

base_model = prepare_model_for_kbit_training(base_model)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=CONFIG.lora_r,
    lora_alpha=CONFIG.lora_alpha,
    lora_dropout=CONFIG.lora_dropout,
    target_modules=list(CONFIG.target_modules),
    bias="none",
    modules_to_save=["score", "classifier"],
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

## 9. Fine-Tuning

In [ ]:
class RerankerDataset(torch.utils.data.Dataset):
    def __init__(self, pairs, tokenizer, max_length=512):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        item = self.pairs[idx]
        encoding = self.tokenizer(
            item["query"],
            item["passage"],
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )
        encoding["labels"] = float(item["label"])
        return encoding

train_dataset = RerankerDataset(train_pairs, tokenizer, CONFIG.max_length)
val_dataset   = RerankerDataset(val_pairs, tokenizer, CONFIG.max_length)

class RerankerTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.view(-1)
        loss_fct = torch.nn.BCEWithLogitsLoss()
        loss = loss_fct(logits, labels.to(logits.dtype))
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=CONFIG.output_dir,
    num_train_epochs=CONFIG.num_train_epochs,
    per_device_train_batch_size=CONFIG.per_device_train_batch_size,
    per_device_eval_batch_size=CONFIG.per_device_eval_batch_size,
    gradient_accumulation_steps=CONFIG.gradient_accumulation_steps,
    learning_rate=CONFIG.learning_rate,
    weight_decay=CONFIG.weight_decay,
    warmup_ratio=CONFIG.warmup_ratio,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none",
    seed=CONFIG.seed,
)

trainer = RerankerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

trainer.train()

## 10 & 11. Test Evaluation: Base vs Fine-Tuned (QLoRA)

In [ ]:
def score_pairs_batch(model_to_test, pairs, tokenizer, batch_size=16):
    model_to_test.eval()
    all_scores = []
    device = next(model_to_test.parameters()).device
    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch_pairs = pairs[i:i + batch_size]
            inputs = tokenizer(
                [p[0] for p in batch_pairs],
                [p[1] for p in batch_pairs],
                padding=True,
                truncation=True,
                max_length=CONFIG.max_length,
                return_tensors="pt",
            ).to(device)
            logits = model_to_test(**inputs).logits.view(-1)
            scores = torch.sigmoid(logits).cpu().tolist()
            all_scores.extend(scores)
    return all_scores

def evaluate_ranking(model_to_test, test_query_records, tokenizer):
    mrr_list, r1_list, r5_list, ndcg5_list = [], [], [], []
    for item in test_query_records:
        query = item["query"]
        pos_texts = item["pos_texts"]
        neg_texts = item["neg_texts"]
        candidates = [(query, text) for text in pos_texts] + [(query, text) for text in neg_texts]
        labels = [1] * len(pos_texts) + [0] * len(neg_texts)
        
        scores = score_pairs_batch(model_to_test, candidates, tokenizer)
        ranked_indices = np.argsort(scores)[::-1]
        ranked_labels = [labels[idx] for idx in ranked_indices]
        
        # MRR
        first_pos = next((r + 1 for r, l in enumerate(ranked_labels) if l == 1), 0)
        mrr_list.append(1.0 / first_pos if first_pos > 0 else 0.0)
        r1_list.append(1.0 if any(l == 1 for l in ranked_labels[:1]) else 0.0)
        r5_list.append(1.0 if any(l == 1 for l in ranked_labels[:5]) else 0.0)
        
        # nDCG@5
        dcg5 = sum((2**l - 1) / math.log2(r + 2) for r, l in enumerate(ranked_labels[:5]))
        ideal = sorted(labels, reverse=True)[:5]
        idcg5 = sum((2**l - 1) / math.log2(r + 2) for r, l in enumerate(ideal))
        ndcg5_list.append(dcg5 / idcg5 if idcg5 > 0 else 0.0)
        
    return {
        "mrr": float(np.mean(mrr_list)),
        "recall_at_1": float(np.mean(r1_list)),
        "recall_at_5": float(np.mean(r5_list)),
        "ndcg_at_5": float(np.mean(ndcg5_list)),
    }

ft_metrics = evaluate_ranking(model, test_queries, tokenizer)

base_eval_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG.base_model_name,
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
)
base_metrics = evaluate_ranking(base_eval_model, test_queries, tokenizer)
del base_eval_model
torch.cuda.empty_cache()

print("=" * 70)
print(f"{'Metric':<15} | {'Base Reranker':<15} | {'Fine-Tuned (QLoRA)':<20} | {'Delta':<10}")
print("-" * 70)
for m in ["mrr", "recall_at_1", "recall_at_5", "ndcg_at_5"]:
    delta = ft_metrics[m] - base_metrics[m]
    print(f"{m:<15} | {base_metrics[m]:<15.4f} | {ft_metrics[m]:<20.4f} | {delta:+.4f}")
print("=" * 70)

## 12 & 13. Save Artifacts & Create Integration Package

In [ ]:
package_dir = Path(CONFIG.package_dir)
adapter_dir = package_dir / "adapter"
tokenizer_dir = package_dir / "tokenizer"

if package_dir.exists():
    shutil.rmtree(package_dir)
package_dir.mkdir(parents=True, exist_ok=True)
adapter_dir.mkdir(parents=True, exist_ok=True)
tokenizer_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(tokenizer_dir))

eval_summary = {
    "base_model": CONFIG.base_model_name,
    "fine_tuning_method": "QLoRA (4-bit NF4 + PEFT LoRA)",
    "metrics_comparison": {
        "base": base_metrics,
        "fine_tuned_qlora": ft_metrics,
    }
}
with open(package_dir / "evaluation_results.json", "w") as f:
    json.dump(eval_summary, f, indent=2)

with open(package_dir / "training_config.json", "w") as f:
    json.dump(CONFIG.__dict__, f, indent=2)

shutil.make_archive("/kaggle/working/reranker_phase2", "zip", package_dir)
print("Artifact package ready at /kaggle/working/reranker_phase2.zip")